### Tutorial 14_B: Task 1 & 3: Dataset Preparation and Hyperparameter Configuration
A baby names dataset to build a One-to-Many RNN and then adjust the hyperparameters below.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers

# --- Hyperparameters (Task 3) ---
UNITS = 128         # Modified from previous state
EPOCHS = 30         # Modified from previous state
LEARNING_RATE = 0.005 # Modified from previous state

# Load dataset from online repository (Social Security Administration baby names via raw github)
url = 'https://raw.githubusercontent.com/hadley/data-baby-names/master/baby-names.csv'
df_names = pd.read_csv(url)
names = df_names['name'].unique().astype(str)

# Preprocessing: Add start and end tokens
names = ['[' + name.lower() + ']' for name in names if len(name) > 2]

# Character mapping
chars = sorted(list(set(''.join(names))))
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}
n_vocab = len(chars)
max_len = max([len(name) for name in names])

print(f'Total unique names: {len(names)}')
print(f'Vocabulary size: {n_vocab}')

Total unique names: 6772
Vocabulary size: 28


### Task 2: Develop a One-to-Many RNN Model
In this 'One-to-Many' started character and the model generates the sequence.

In [ ]:
# Prepare training data (Input: name[i], Output: name[i+1])
X_data = []
y_data = []

for name in names[:5000]: # Using subset for speed
    for i in range(len(name) - 1):
        X_data.append(char_to_int[name[i]])
        y_data.append(char_to_int[name[i+1]])

X = tf.one_hot(X_data, n_vocab)
y = tf.one_hot(y_data, n_vocab)

# Build Model
model = tf.keras.Sequential([
    layers.Reshape((1, n_vocab), input_shape=(n_vocab,)),
    layers.SimpleRNN(UNITS, return_sequences=False),
    layers.Dense(n_vocab, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

# Train
model.fit(X, y, epochs=EPOCHS, batch_size=128, verbose=1)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


268/268 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2.5306
Epoch 2/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4583
Epoch 3/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 2.4530
Epoch 4/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 2.4515
Epoch 5/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.4484
Epoch 6/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4494
Epoch 7/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4503
Epoch 8/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4486
Epoch 9/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4495
Epoch 10/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4498
Epoch 11/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4481
Epoch 12/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4479
Epoch 13/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4486
Epoch 14/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4484
Epoch 15/30
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4499


In [ ]:
def generate_name(seed_char='['):
    generated = seed_char
    for _ in range(max_len):
        x_pred = tf.one_hot([char_to_int[generated[-1]]], n_vocab)
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = np.random.choice(len(preds), p=preds)
        next_char = int_to_char[next_index]

        if next_char == ']':
            break
        generated += next_char
    return generated.replace('[', '').capitalize()

print("Generated Baby Names:")
for _ in range(5):
    print(generate_name())

Generated Baby Names:
Clele
Vonoyauano
R
Clen
Ck
